In [ ]:
%matplotlib widget
import matplotlib.pyplot as plt
import numpy as np
from astropy.io import fits

In [ ]:
# reset defalult plotting values
plt.rcParams['figure.figsize'] = (7, 7)
plt.rc('font', family='sans-serif')
plt.rc('axes', labelsize=14)
plt.rc('axes', labelweight='bold')
plt.rc('axes', titlesize=16)
plt.rc('axes', titleweight='bold')

# ASTR 350: Astronomical Techniques
# Lecture 11


### Prof. Robert Quimby <br> February 24, 2026

&copy; 2026 Robert Quimby

## Review Quiz 8

In [ ]:
def remove_overscan(rawimage):
    """
    return the active CCD region with the mean overscan value subtracted
    
    `rawimage` is a 2D numpy array
    """
    
    # number of rows in the active region 
    # (assume the active region is square and the rawimage has overscan columns but not rows)
    nrows = ????
    
    # get a 2D slice of the overscan region
    overscan = ????
    
    # get a 2D slice of the active region
    active_region = ????

    # return the active region with the mean overscan value subtracted
    return ????

## Image Processing

- [notebook](../tutorials/image.processing.ipynb) 
- [video](https://youtu.be/I86Ov1W9aEY) (28 min)

## Bias Exposures

* If we read out the CCD without exposing (e.g., a zero second exposure with the shutter closed), $N_{\gamma}$ will be zero so the measured signal is just the sum of the bias terms (time dependant and time independant, to first order).
$$C = { B_f + B_s + \epsilon N_{\gamma} \over g} \rightarrow { B_f \over g} + {B_s \over g}$$
* we can subtract off the median overscan value to recover the time independant (to first order) bias.

Note that when we read out a bias exposure, we get the gain divided values (e.g., ${ B_f \over g}$, not $B_f$)

## How to Construct a Master Bais Frame

Master bias frames are used to remove the static bias from the image

* record ~25 bias frames
* subtract off the overscan from each bias frame
* **for each pixel** on the CCD, determine the median, overscan subtracted value using all the bias frames

## Lets work with some real data on GitHub

If you have not already, clone the data in https://github.com/SDSU-astr350-2026-spring/20170928
- In a Unix terminal, type:<pre>
cd
mkdir data
cd data
git clone https://github.com/SDSU-astr350-2026-spring/20170928.git
</pre>

## Make a list of paths to the bias frames

In [ ]:
# locate the bias frames
import ????
datadir = '../data/20170928/'
paths = []
for i in range (????):
    fname = 'a{:03d}.fit'.format(i)
    path = ????
    paths.append(path)
paths

## Load all bias images into a 3D array--a "datacube"

In [ ]:
# build the data cube
images = []
for path in paths:
    # load the image data
    rawimage = fits.getdata(????)

    # remove the overscan
    image = remove_overscan(????)

    # add to the list
    ????
    
# convert images list into a 3D array
datacube = ????

In [ ]:
datacube.shape

## Make the Master Bias

In [ ]:
# get the typical value of each pixel accross all images
bias = ????

In [ ]:
# show the master bias image
mean, std = bias.mean(), bias.std()
vmin = mean - std
vmax = mean + 5 * std
plt.figure()
plt.imshow(bias, vmin=vmin, vmax=vmax);

## What we ultimately want is (values proportional to) $N_{\gamma}$

$$ N_{\gamma} = {gC - B_f - B_s \over \epsilon}$$

Thus we need to determine each of the other parameters

* $g$ is set in the lab and can be measured from the photon transfer curve
* $C$ is the raw pixel value (ADU)
* $B_f$ can be measured from the overscan
* $B_s$ can be measured from (overscan subtracted) zero second dark exposures
* but $\epsilon$ is difficult to measure directly

* however, all we need is the relative, pixel to pixel efficiency, $f$.

$$f = {\epsilon \over \bar{\epsilon}}$$

## Correcting for pixel-to-pixel variations (Flat-Fielding)

- need a master flat-field image with each pixel value proportional to the efficiency
- similar (but not identical!) to construction of the master bias

## Detrending data

Remove bias and pixel-to-pixel variations to determine the relative numbers of photons incident on each pixel

Process:
* remove overscan
* trim to active area
* subtract master bias
* divide by master flat